## loading data

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tarfile
from pathlib import Path
import urllib.request

In [2]:
def load_data():
    root = 'https://spamassassin.apache.org/old/publiccorpus/'
    spam = root  + "20030228_spam.tar.bz2"
    ham = root + "20030228_easy_ham.tar.bz2"
    
    spam_path = Path() / 'datasets' / 'spam'
    spam_path.mkdir(exist_ok=True, parents=True)
    for dir_name, tar_name, url in  (('easy_gam', 'ham' ,ham), ('spam', 'spam', spam)):
        if not (spam_path / dir_name).is_dir():
            path = (spam_path / tar_name).with_suffix('.tar.bz2')
            print('Downloading ' + str(path))
            urllib.request.urlretrieve(url, path)
            tar_bz2_file = tarfile.open(path)
            tar_bz2_file.extractall(path=spam_path, filter='data')
            tar_bz2_file.close()
    return [spam_path / dir_name for dir_name in ('easy_ham', 'spam')]

In [3]:
ham_dir, spam_dir = load_data()

In [4]:
ham_filenames = [f for f in sorted(ham_dir.iterdir()) if len(f.name) > 20]
spam_filenames = [f for f in sorted(spam_dir.iterdir()) if len(f.name) > 20]

In [5]:
print('ham:', len(ham_filenames), 'spam:', len(spam_filenames))

ham: 2500 spam: 500


In [6]:
import email
import email.policy

def load_email(file_name):
    with open(file_name, 'rb') as f:
        return email.parser.BytesParser(policy=email.policy.default).parse(f)

In [7]:
ham_emails = [load_email(filename) for filename in ham_filenames]
spam_emails = [load_email(filename) for filename in spam_filenames]

In [8]:
print(ham_emails[1].get_content().strip())

Martin A posted:
Tassos Papadopoulos, the Greek sculptor behind the plan, judged that the
 limestone of Mount Kerdylio, 70 miles east of Salonika and not far from the
 Mount Athos monastic community, was ideal for the patriotic sculpture. 
 
 As well as Alexander's granite features, 240 ft high and 170 ft wide, a
 museum, a restored amphitheatre and car park for admiring crowds are
planned
---------------------
So is this mountain limestone or granite?
If it's limestone, it'll weather pretty fast.

------------------------ Yahoo! Groups Sponsor ---------------------~-->
4 DVDs Free +s&p Join Now
http://us.click.yahoo.com/pt6YBB/NXiEAA/mG3HAA/7gSolB/TM
---------------------------------------------------------------------~->

To unsubscribe from this group, send an email to:
forteana-unsubscribe@egroups.com

 

Your use of Yahoo! Groups is subject to http://docs.yahoo.com/info/terms/


In [9]:
print(spam_emails[3].get_content().strip())

##################################################
#                                                #
#                 Adult Club                     #
#           Offers FREE Membership               #
#                                                #
##################################################

>>>>>  INSTANT ACCESS TO ALL SITES NOW
>>>>>  Your User Name And Password is.
>>>>>  User Name: zzzz@spamassassin.taint.org
>>>>>  Password: 760382

5 of the Best Adult Sites on the Internet for FREE!
---------------------------------------
NEWS 08/18/02
With just over 2.9 Million Members that signed up for FREE, Last month there were 721,184 New
Members. Are you one of them yet???
---------------------------------------
Our Membership FAQ

Q. Why are you offering free access to 5 adult membership sites for free?
A. I have advertisers that pay me for ad space so you don't have to pay for membership.

Q. Is it true my membership is for life?
A. Absolutely you'll never have to pay a cen

In [10]:
def get_email_structure(email):
    if isinstance(email, str):
        return email
    payload = email.get_payload()
    if isinstance(payload, list):
        multipart = ','.join([get_email_structure(submail) for submail in payload])
        return f"multipart({multipart})"
    else:
        return email.get_content_type()

In [11]:
from collections import Counter

def count_structures(emails):
    structures = Counter()
    for email in emails:
        structure = get_email_structure(email)
        structures[structure] += 1
        
    return structures

In [12]:
count_structures(ham_emails).most_common()

[('text/plain', 2408),
 ('multipart(text/plain,application/pgp-signature)', 66),
 ('multipart(text/plain,text/html)', 8),
 ('multipart(text/plain,text/plain)', 4),
 ('multipart(text/plain)', 3),
 ('multipart(text/plain,application/octet-stream)', 2),
 ('multipart(text/plain,text/enriched)', 1),
 ('multipart(text/plain,application/ms-tnef,text/plain)', 1),
 ('multipart(multipart(text/plain,text/plain,text/plain),application/pgp-signature)',
  1),
 ('multipart(text/plain,video/mng)', 1),
 ('multipart(text/plain,multipart(text/plain))', 1),
 ('multipart(text/plain,application/x-pkcs7-signature)', 1),
 ('multipart(text/plain,multipart(text/plain,text/plain),text/rfc822-headers)',
  1),
 ('multipart(text/plain,multipart(text/plain,text/plain),multipart(multipart(text/plain,application/x-pkcs7-signature)))',
  1),
 ('multipart(text/plain,application/x-java-applet)', 1)]

In [13]:
count_structures((spam_emails)).most_common()

[('text/plain', 218),
 ('text/html', 183),
 ('multipart(text/plain,text/html)', 45),
 ('multipart(text/html)', 20),
 ('multipart(text/plain)', 19),
 ('multipart(multipart(text/html))', 5),
 ('multipart(text/plain,image/jpeg)', 3),
 ('multipart(text/html,application/octet-stream)', 2),
 ('multipart(text/plain,application/octet-stream)', 1),
 ('multipart(text/html,text/plain)', 1),
 ('multipart(multipart(text/html),application/octet-stream,image/jpeg)', 1),
 ('multipart(multipart(text/plain,text/html),image/gif)', 1),
 ('multipart/alternative', 1)]

In [14]:
for header, value in spam_emails[0].items():
    print(header, ":", value)

Return-Path : <12a1mailbot1@web.de>
Delivered-To : zzzz@localhost.spamassassin.taint.org
Received : from localhost (localhost [127.0.0.1])	by phobos.labs.spamassassin.taint.org (Postfix) with ESMTP id 136B943C32	for <zzzz@localhost>; Thu, 22 Aug 2002 08:17:21 -0400 (EDT)
Received : from mail.webnote.net [193.120.211.219]	by localhost with POP3 (fetchmail-5.9.0)	for zzzz@localhost (single-drop); Thu, 22 Aug 2002 13:17:21 +0100 (IST)
Received : from dd_it7 ([210.97.77.167])	by webnote.net (8.9.3/8.9.3) with ESMTP id NAA04623	for <zzzz@spamassassin.taint.org>; Thu, 22 Aug 2002 13:09:41 +0100
From : 12a1mailbot1@web.de
Received : from r-smtp.korea.com - 203.122.2.197 by dd_it7  with Microsoft SMTPSVC(5.5.1775.675.6);	 Sat, 24 Aug 2002 09:42:10 +0900
To : dcek1a1@netsgo.com
Subject : Life Insurance - Why Pay More?
Date : Wed, 21 Aug 2002 20:31:57 -1600
MIME-Version : 1.0
Message-ID : <0103c1042001882DD_IT7@dd_it7>
Content-Type : text/html; charset="iso-8859-1"
Content-Transfer-Encoding : qu

In [15]:
spam_emails[0]["Subject"]

'Life Insurance - Why Pay More?'

### spliting train and test sets 

In [16]:
from sklearn.model_selection import train_test_split

X = np.array(ham_emails + spam_emails, dtype=object)
y = np.array([0] * len(ham_emails) + [1] * len(spam_emails))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=52)

In [17]:
import re
from html import unescape

def html_to_text(html):
    text = re.sub(r'<head.*?>.*?</head>', "", html, flags=re.M | re.S | re.I)
    text = re.sub(r'<a\s.*?>', 'HYPERLINK', text, flags=re.M | re.S | re.I)
    text = re.sub(r'<.*?>', '', text, flags=re.M | re.S)
    text = re.sub(r'(\s*\n)', "\n", text, flags=re.M | re.S)
    return unescape(text)
    

In [20]:
html_spam_emails = [email for email in X_train[y_train==1] if get_email_structure(email) == 'text/html']
sample_spam_html = html_spam_emails[7]

print(sample_spam_html.get_content().strip()[:1000], '...')

<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.0 Transitional//EN">
<HTML><HEAD><TITLE>EM049</TITLE>
</HEAD>
<BODY bgColor="#FFFFFF" leftMargin="0" topMargin="0" MARGINHEIGHT="0" MARGINWIDTH="0">
<table width="550" border="0" cellspacing="0" cellpadding="0" align="center">
  <TBODY>
  <TR>
    <TD colSpan=2><A href="http://theadmanager.com/server/c.asp?ad_key=FWOATHSKUDMM&ext=1">
	<IMG src="http://admanmail.com/ads/adman/kmart/EM049_01.gif" alt="" border="0" height="59" width="550"></A></TD></TR>
  <TR>
    <TD><A href="http://theadmanager.com/server/c.asp?ad_key=FWOATHSKUDMM&ext=1">
	<IMG src="http://admanmail.com/ads/adman/kmart//EM049_02.gif" alt="" border="0" height="83" width="336"></A></TD>
    <TD><A href="http://theadmanager.com/server/c.asp?ad_key=FWOATHSKUDMM&ext=1">
	<IMG src="http://admanmail.com/ads/adman/kmart//EM049_03.gif" alt="" border=0 height=83 width=214></A></TD></TR>
  <TR>
    <TD><A href="http://theadmanager.com/server/c.asp?ad_key=FWOATHSKUDMM&ext=1">
	<IMG src="htt

In [23]:
print(html_to_text(sample_spam_html.get_content())[:1000], '...')


    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
You are receiving this mailing because you are a
member of SendGreatOffers.com and subscribed as:JM@NETNOTEINC.COM
To unsubscribe HYPERLINK
Click Here
(http://admanmail.com/subscription.asp?em=JM@NETNOTEINC.COM&l=SGO)
or reply to this email with REMOVE in the subject line - you must
also include the body of this message to be unsubscribed. Any correspondence about
the products/services should be directed to
the company in the ad.
%EM%JM@NETNOTEINC.COM%/EM%
 ...


In [24]:
def email_to_text(email):
    html=None
    for part in email.walk():
        ctype = part.get_content_type()
        if not ctype in ('text/plain', 'text/html'):
            continue
        try:
            content = part.get_content()
        except:
            content = str(part.get_payload())
        if ctype == 'text/plain':
            return content
        else:
            html = content
    if html:
        return html_to_text(html)

In [27]:
print(email_to_text(sample_spam_html)[:500], '...')


    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
    HYPERLINK
You are receiving this mailing because you are a
member of SendGreatOffers.com and subscribed as:JM@NETNOTEINC.COM
To unsubscribe HYPERLINK
Click Here
(http://admanmail.com/subscription.asp?em=JM@NETNOTEINC.COM&l=SGO)
or reply to this email with REMOVE in the subject line - you must
also include the body of this message to be unsubscribed. Any correspondenc ...


In [30]:
import nltk

stemmer = nltk.PorterStemmer()

for word in ["Computations", "Computation", "Computing", "Computed", "Compute",
             "Compulsive"]:
    print(word, '=>', stemmer.stem(word))

Computations => comput
Computation => comput
Computing => comput
Computed => comput
Compute => comput
Compulsive => compuls


In [33]:
import urlextract

url_extractor = urlextract.URLExtract()

some_text = "Will it detect github.com and https://youtu.be/7Pq-S557XQU?t=3m32s"
print(url_extractor.find_urls(some_text))

['github.com', 'https://youtu.be/7Pq-S557XQU?t=3m32s']


In [63]:
from sklearn.base import BaseEstimator, TransformerMixin

class EmailToWordCounterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, strip_headers=True, remove_punctuation=True, lower_case=True, 
                 replace_urls=True, replace_numbers=True, stemming=True):
        self.strip_headers = strip_headers
        self.remove_punctuation = remove_punctuation
        self.lower_case = lower_case
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.stemming = stemming
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        X_transformed = []
        
        for email in X:
            text = email_to_text(email) or ""
            
            if self.lower_case:
                text = text.lower()
            if self.replace_urls and url_extractor is not None:
                urls = list(set(url_extractor.find_urls(text)))
                urls.sort(key=lambda url: len(url), reverse=True)
                for url in urls:
                    text = text.replace(url, " URL ")
            if self.replace_numbers:
                text = re.sub(r'\d+(?:\.\d*)?(?:[eE][+-]?\d+)?', 'NUMBER', text)
            if self.remove_punctuation:
                text = re.sub(r'\W+', ' ', text, flags=re.M)
            word_counts = Counter(text.split())
            if self.stemming and stemmer is not None:
                stemmed_word_counts = Counter()
                for word, count in word_counts.items():
                    stemmed_word = stemmer.stem(word)
                    stemmed_word_counts[stemmed_word] += count
                word_counts = stemmed_word_counts
            X_transformed.append(word_counts)
        return np.array(X_transformed)
                    

In [73]:
X_few_wordcounts = EmailToWordCounterTransformer().fit_transform(X_train[:3])
X_few_wordcounts

array([Counter({'number': 6, 'and': 4, 'i': 3, 'the': 3, 'you': 3, 'url': 2, 'write': 2, 'on': 2, 'a': 2, 'date': 1, 'numbertnumb': 1, 'jeremi': 1, 'about': 1, 'peopl': 1, 'who': 1, 'ignor': 1, 'basic': 1, 'languag': 1, 'rule': 1, 'entir': 1, 'agre': 1, 'with': 1, 'him': 1, 'how': 1, 'r': 1, 'u': 1, 'numberday': 1, 'is': 1, 'best': 1, 'way': 1, 'to': 1, 'make': 1, 'me': 1, 'shift': 1, 'my': 1, 'attent': 1, 'respect': 1, 'away': 1, 'from': 1, 'realli': 1, 'fast': 1, 'anoth': 1, 'pet': 1, 'peev': 1, 'have': 1, 'onli': 1, 'been': 1, 'speak': 1, 'english': 1, 'regular': 1, 'basi': 1, 'for': 1, 'bit': 1, 'more': 1, 'than': 1, 'three': 1, 'year': 1, 'even': 1, 'can': 1, 'grok': 1, 'differ': 1, 'between': 1, 'are': 1, 'your': 1, 'as': 1, 'mjd': 1, 'wrote': 1, 'clpm': 1, 'said': 1, 'in': 1, 'yapc': 1, 'movi': 1}),
       Counter({'number': 9, 'i': 8, 'to': 6, 'the': 5, 'it': 4, 'but': 3, 'fork': 3, 'com': 3, 'beberg': 3, 'for': 3, 'o': 3, 'async': 2, 'io': 2, 'on': 2, 'in': 2, 'that': 2, 'way'

In [68]:
from scipy.sparse import csr_matrix

class WordCounterToVectorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vocabulary_size=1000):
        self.vocabulary_size = vocabulary_size
        
    def fit(self, X, y=None):
        total_count = Counter()
        for word_count in X:
            for word, count in word_count.items():
                total_count[word] += min(count, 10)
        most_common = total_count.most_common()[:self.vocabulary_size]
        self.vocabulary_ = {word : index+1 for index, (word, count) in enumerate(most_common)}
        return self
    
    def transform(self, X, y=None):
        rows = []
        cols = []
        data = []
        for row, word_count in enumerate(X):
            for word, count in word_count.items():
                rows.append(row)
                cols.append(self.vocab_.get(word, 0))
                data.append(count)
        return csr_matrix((data, (rows, cols)), shape=(len(X), self.vocabulary_size+1))

In [77]:
vocab_transformer = WordCounterToVectorTransformer(vocabulary_size=10)
(vocab_transformer.fit_transform(X_few_wordcounts)).toarray()

array([[ 65,   6,   3,   3,   1,   2,   4,   3,   2,   1,   0],
       [156,   9,   8,   5,   6,   1,   1,   1,   2,   3,   4],
       [  2,   6,   0,   0,   0,   3,   0,   0,   0,   0,   0]])

In [80]:
vocab_transformer.vocabulary_

{'number': 1,
 'i': 2,
 'the': 3,
 'to': 4,
 'url': 5,
 'and': 6,
 'you': 7,
 'on': 8,
 'for': 9,
 'it': 10}

## Giving prepared data to a model

In [88]:
from sklearn.pipeline import Pipeline

preprocessing = Pipeline([('email_to_text', EmailToWordCounterTransformer()), ('text_to_vector', WordCounterToVectorTransformer())])

X_train_transformed = preprocessing.fit_transform(X_train)

In [89]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

log_reg = LogisticRegression(max_iter=1000,random_state=52)

score = cross_val_score(log_reg, X_train_transformed, y_train, cv=3, scoring='accuracy')
print(score)
print(score.mean())

[0.9975  0.98375 0.98   ]
0.9870833333333334


In [90]:
from sklearn.metrics import precision_score, recall_score

X_test_transformed = preprocessing.transform(X_test)

log_reg.fit(X_train_transformed, y_train)

y_pred = log_reg.predict(X_test_transformed)

print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))

precision: 0.9883720930232558
recall: 0.9444444444444444
